# Phase 3 — Alpha Calibration Notebook

This notebook sweeps the ensemble weight `alpha` (SVM weight in the hybrid)
over `[0.0, 1.0]` on the `test_ua` validation split and plots the resulting
F1-Macro curve.

## Key question
What is the optimal balance between SVM (Model A) and Calibrated SBERT (Model B)?

## Phase 2 finding
Phase 2 calibration showed alpha=1.0 (pure SVM) was optimal because SBERT
thresholds were hand-tuned, producing F1=0.30 standalone.

## Phase 3 hypothesis
With data-driven calibrated thresholds, SBERT should contribute positively at
some alpha < 1.0, making the hybrid genuinely synergistic.

In [ ]:
import sys, os
# Navigate to repo root from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PHASE3    = os.path.join(REPO_ROOT, 'phase3')
PHASE2    = os.path.join(REPO_ROOT, 'phase2')
for p in [REPO_ROOT, PHASE3, PHASE2]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

# Phase 3 imports
import config
import utils as p3_utils
from grading.classical_grader import ClassicalGrader
from grading.calibrated_scorer import CalibratedScorer
from grading.hybrid_grader import SynergisticHybridGrader

print('Imports OK')

In [ ]:
# ── Step 1: Load dataset ────────────────────────────────────────────────────
print('Loading SciEntsBank...')
dataset = p3_utils.load_scientsbank()
dataset = p3_utils.convert_labels(dataset, config.LABEL_SCHEME)

X_train, X_val, y_train, y_val, train_ref, val_ref = p3_utils.prepare_dataframe(dataset, 'test_ua')
X_val_list   = X_val.tolist()
val_ref_list = val_ref.tolist()
y_val_list   = y_val.tolist()
print(f'Validation split (test_ua): {len(X_val_list)} samples')

In [ ]:
# ── Step 2: Load models ─────────────────────────────────────────────────────
print('Loading ClassicalGrader (Phase 2 SVM)...')
classical = ClassicalGrader()
classical.load(config.MODEL_A_PATH)
print('✓ SVM loaded')

print('Loading CalibratedScorer...')
calibrated = CalibratedScorer(model_name=config.SBERT_MODEL)
if not calibrated.load_thresholds():
    print('Thresholds not found — running calibration (this takes a few minutes)...')
    X_tr, _, y_tr, _, tr_ref, _ = p3_utils.prepare_dataframe(dataset, 'train')
    calibrated.calibrate(X_tr, tr_ref, y_tr)
print(f'✓ Calibrated thresholds: {calibrated.thresholds}')

In [ ]:
# ── Step 3: Alpha sweep ─────────────────────────────────────────────────────
alphas    = np.arange(0.0, 1.05, 0.05)
f1_scores = []

LABEL_MAP = {'correct': 0, 'incorrect': 1, 'partially correct': 2}

print(f'Running alpha sweep ({len(alphas)} steps)...')
for alpha in alphas:
    hybrid = SynergisticHybridGrader(
        classical_grader=classical,
        calibrated_scorer=calibrated,
        alpha=float(alpha),
    )
    preds = [
        hybrid.grade(s, r)['label_idx']
        for s, r in zip(X_val_list, val_ref_list)
    ]
    f1 = f1_score(y_val_list, preds, average='macro', zero_division=0)
    f1_scores.append(f1)
    print(f'  alpha={alpha:.2f}  F1={f1:.4f}')

optimal_alpha = alphas[np.argmax(f1_scores)]
optimal_f1    = max(f1_scores)
print(f'\nOptimal alpha = {optimal_alpha:.2f}  (F1={optimal_f1:.4f})')

In [ ]:
# ── Step 4: Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(alphas, f1_scores,
        marker='o', color='royalblue',
        linewidth=2, markersize=6, label='F1 Macro')

ax.axvline(optimal_alpha, color='red', linestyle='--', linewidth=1.5,
           label=f'Optimal α={optimal_alpha:.2f} (F1={optimal_f1:.3f})')

# Mark Phase 2 default alpha=0.4
p2_alpha = 0.4
p2_f1    = f1_scores[int(round(p2_alpha / 0.05))]
ax.axvline(p2_alpha, color='orange', linestyle=':', linewidth=1.5,
           label=f'Phase 2 default α={p2_alpha} (F1={p2_f1:.3f})')

ax.set_xlabel('Alpha (SVM Weight)', fontsize=12)
ax.set_ylabel('F1 Macro on Validation Set', fontsize=12)
ax.set_title('Phase 3: Alpha Calibration (SVM ↔ Calibrated SBERT)', fontsize=13, fontweight='bold')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 0.80)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()

save_path = os.path.join(PHASE3, 'notebooks', 'alpha_calibration_p3.png')
plt.savefig(save_path, dpi=150)
plt.show()
print(f'Plot saved → {save_path}')

In [ ]:
# ── Step 5: Interpretation ────────────────────────────────────────────────────
svm_weight  = optimal_alpha * 100
sbert_weight = (1 - optimal_alpha) * 100

print('═' * 55)
print('ALPHA CALIBRATION INTERPRETATION')
print('═' * 55)
print(f'Optimal alpha  = {optimal_alpha:.2f}')
print(f'  SVM weight   = {svm_weight:.0f}%')
print(f'  SBERT weight = {sbert_weight:.0f}%')
print()
print('Phase 2 comparison:')
print(f'  Phase 2 default alpha = 0.4 → F1 = {p2_f1:.4f}')
print(f'  Phase 3 optimal alpha = {optimal_alpha:.2f} → F1 = {optimal_f1:.4f}')
print(f'  Delta: {(optimal_f1 - p2_f1)*100:+.1f}pp')
print()
if optimal_alpha < 0.95:
    print('✓ Calibrated SBERT contributes positively to the ensemble.')
    print('  Phase 3 fixed the root cause: data-driven thresholds make')
    print('  SBERT a genuine contributor, not a drag.')
else:
    print('→ SVM still dominates (alpha near 1.0).')
    print('  The gating mechanism still provides value by saving SBERT')
    print('  computation on high-confidence predictions.')
print('═' * 55)